In [65]:
import random
import copy

In [72]:
COLOURS = ['Red', 'Blue', 'Green', 'Yellow']
NUMBERS = list(range(10))         
PLAYERS = ['p1', 'p2', 'p3']  
PLAYER_NAMES = {'p1': 'P1 (Minimax Defensive)',
    'p2': 'P2 (Expectimax Offensive)' ,
    'p3': 'P3',
}

DEFAULT_SEED = 42

In [73]:
class Card:# single uno card

    def __init__(self, colour, value):
        self.colour = colour
        self.value = value           

        
    def __repr__(self):
        return f"{self.colour} {self.value}"

    def __eq__(self, other):
        return (isinstance(other, Card)
                and self.colour == other.colour
                and self.value == other.value)

    def __hash__(self):
        return hash((self.colour, self.value))

    def __deepcopy__(self, memo):
        #colour value dono fixed tu no need to recurse
        return Card(self.colour, self.value)



    def matches(self, top: 'Card'): #same colour or number
        return self.colour == top.colour or self.value == top.value
        
    def is_skip(self):
        return self.value == 'Skip'

#DEBUG
c1 = Card('Red' , 8)
print(c1.is_skip())

c2 = Card('Yellow ', 'Skip')
print(c2.is_skip())

c1.__repr__()

False
True


'Red 8'

In [74]:
def deck_generator(): #total 4×11 so 44cards 0 say 9 sab mai and 1 skip bhi so 11 

    deck = []
    for colour in COLOURS:
        for num in NUMBERS:                  
            deck.append(Card(colour, num))
        deck.append(Card(colour, 'Skip'))   

        
    random.shuffle(deck)
    return deck



#DEBUG
#deck= deck_generator()
#print(deck)
#len(deck)

In [75]:
def get_valid_moves(hand, top_card):
    return [card for card in hand if card.matches(top_card)]

#DEBUG
hand = [Card('Red', 6) , Card('Blue', 3), Card('Yellow', 4) , Card('Yellow', 'Skip') , Card('Green', 'Skip') ]
top = Card('Yellow', 6)
get_valid_moves(hand , top)

[Red 6, Yellow 4, Yellow Skip]

In [76]:
def apply_move(state, move):#player key is Player Names ki keys p1 ,p2 , p3

    player_key, card = move
    
    state_updated = copy.deepcopy(state)#deep copy takay original wala is not mutated
    skip_player= None

    if card is None:#card nai hai valid also reshuffle already played ya discards into deck agar deck hi khali hogaya
        if not state_updated['deck'] and state_updated.get('discards'):
            state_updated['deck'] = state_updated['discards']
            random.shuffle(state_updated['deck'])
            state_updated['discards'] = []

        
        if state_updated['deck']:
            drawn = state_updated['deck'].pop(0)
            state_updated[player_key].append(drawn)
       

    else:
        hand = state_updated[player_key]
        for i, c in enumerate(hand):
            if c == card:
                hand.pop(i)
                break

        if 'discards' not in state_updated:#purana top ko discard mai 
            state_updated['discards'] = []
            
        state_updated['discards'].append(state_updated['top_card'])

        state_updated['top_card'] = card


        
        if card.is_skip():
            index = PLAYERS.index(player_key)
            skip_player = PLAYERS[(index + 1) % len(PLAYERS)]


    
    return state_updated, skip_player